# 🏥 KG-CMI: Knowledge Graph Enhanced Cross-Mamba Interaction for Med-VQA
### 🎓 Minor Project Training Notebook (Google Colab / Cloud GPU)

This notebook contains the complete pipeline for setting up, configuring, and training the upgraded **KG-CMI** model with **Relational GCN (R-GCN)** knowledge graph reasoning on Med-VQA datasets (VQA-RAD / SLAKE / OVQA).

## Step 1: Hardware Check & Repository Cloning

In [ ]:
# Verify NVIDIA GPU Availability
!nvidia-smi

# Clone project repository (Public repo URL or fallback)
!git clone https://github.com/Anjali-2807/Med-VQA.git || git clone https://github.com/BioMedIA-repo/KG-CMI.git Med-VQA
%cd Med-VQA

## Step 2: Fast Install Required Dependencies & Pre-built Mamba CUDA Kernels

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets timm einops
!pip install -q torch-geometric

# Fast Mamba & Causal Conv1D pre-built wheel installation (30 seconds)
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

## Step 3: Mount Google Drive for Persistence & Dataset Paths

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define output directory for best model checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## Step 4: Inject Upgraded Relational GCN (R-GCN) KGE Module

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RelationalGraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features, num_relations=4, num_bases=4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_relations = num_relations
        self.num_bases = num_bases

        self.weight_self = nn.Parameter(torch.Tensor(in_features, out_features))
        self.bases = nn.Parameter(torch.Tensor(num_bases, in_features, out_features))
        self.rel_coeffs = nn.Parameter(torch.Tensor(num_relations, num_bases))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_self)
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.rel_coeffs)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type):
        N_v = x.size(0)
        out = torch.matmul(x, self.weight_self)
        W_r = torch.einsum('rb, bio -> rio', self.rel_coeffs, self.bases)
        src, dst = edge_index[0], edge_index[1]

        for r in range(self.num_relations):
            rel_mask = (edge_type == r)
            if not rel_mask.any():
                continue
            r_src, r_dst = src[rel_mask], dst[rel_mask]
            msg = torch.matmul(x[r_src], W_r[r])
            deg = torch.zeros(N_v, device=x.device).scatter_add_(0, r_dst, torch.ones_like(r_dst, dtype=torch.float)).clamp(min=1.0)
            norm = 1.0 / deg[r_dst].unsqueeze(-1)
            out.scatter_add_(0, r_dst.unsqueeze(-1).expand(-1, self.out_features), msg * norm)

        return out + self.bias

print("R-GCN module ready for integration!")

## Step 5: Start Model Training & Evaluation

In [ ]:
# Execute main training script
# Adjust parameters based on target dataset (VQA-RAD / SLAKE / OVQA)
!python main.py \
    --config ./lavis/projects/blip2/train/vqa_rad.yaml \
    --output_dir /content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints